# 面试问题：FlashAttention 为什么既精确又省显存？Online Softmax 和分块 attention 怎样从零实现？

**一句话回答**：它没有把 attention 近似成线性复杂度，而是按 Q/K/V tile 在片上存储中计算，利用 online softmax 的运行最大值与归一化因子，在不保存完整 `N×N` score/probability 的情况下得到与标准 attention 等价的结果，从而减少 HBM 读写。

本 Notebook 用 NumPy 手写稳定 softmax、online recurrence、二维 tiled attention、causal mask、块大小不变性、IO/内存估算和确定性 dropout 思路。


In [ ]:
import hashlib,math  # 导入本单元所需的依赖。
import numpy as np  # 导入本单元所需的依赖。

SEED130=13001; rng130=np.random.default_rng(SEED130)  # 计算并保存当前步骤的中间状态。
assert SEED130==13001  # 用受控断言验证关键不变量。
assert np.exp(-np.inf)==0  # 用受控断言验证关键不变量。
assert np.isfinite(rng130.normal())  # 用受控断言验证关键不变量。


## 1. 先建立数值稳定的标准 attention oracle

标准实现显式形成 `S=QKᵀ/√d`，逐行减最大值后 softmax，再乘 `V`。causal mask 必须在 softmax 前写成负无穷。这个慢但清晰的实现是分块 kernel 的数值 oracle；没有 oracle 就无法判断性能优化是否悄悄改变语义。


In [ ]:
def softmax130(x):  # 定义本节可复用的核心函数。
    m=np.max(x,axis=-1,keepdims=True); e=np.exp(x-m); return e/e.sum(axis=-1,keepdims=True)  # 计算并保存当前步骤的中间状态。
def attention130(q,k,v,causal=False):  # 定义本节可复用的核心函数。
    s=q@k.T/math.sqrt(q.shape[1])  # 计算并保存当前步骤的中间状态。
    if causal: s=np.where(np.arange(len(k))[None,:]<=np.arange(len(q))[:,None],s,-np.inf)  # 按当前条件选择后续控制路径。
    return softmax130(s)@v  # 返回当前分支计算出的结果。
q130=rng130.normal(size=(7,4)); k130=rng130.normal(size=(7,4)); v130=rng130.normal(size=(7,3))  # 计算并保存当前步骤的中间状态。
assert attention130(q130,k130,v130).shape==(7,3)  # 用受控断言验证关键不变量。
assert np.allclose(softmax130(np.array([[1000.,1001.]])),[[1/(1+math.e),math.e/(1+math.e)]])  # 用受控断言验证关键不变量。
assert np.all(np.isfinite(attention130(q130,k130,v130,True)))  # 用受控断言验证关键不变量。


## 2. Online Softmax 可合并任意分块

对已处理元素维护运行最大值 `m`、指数和 `l`。新块最大值到来时，旧累计量乘 `exp(m_old-m_new)` 再合并。这个 rescale 是关键：缺少它时，不同 block 的局部 softmax 无法直接相加，且大 logit 会溢出。


In [ ]:
def online_stats130(blocks):  # 定义本节可复用的核心函数。
    m=-np.inf; l=0.0  # 计算并保存当前步骤的中间状态。
    for x in blocks:  # 遍历输入元素以累积或检查结果。
        bm=float(np.max(x)); nm=max(m,bm)  # 计算并保存当前步骤的中间状态。
        l=l*np.exp(m-nm)+np.exp(x-nm).sum(); m=nm  # 计算并保存当前步骤的中间状态。
    return m,l  # 返回当前分支计算出的结果。
row130=np.array([1000.,998.,1004.,-20.]); m130,l130=online_stats130([row130[:2],row130[2:]])  # 计算并保存当前步骤的中间状态。
assert m130==1004  # 用受控断言验证关键不变量。
assert math.isclose(l130,np.exp(row130-row130.max()).sum())  # 用受控断言验证关键不变量。
assert math.isclose(math.log(l130)+m130,1004+math.log(np.exp(row130-1004).sum()))  # 用受控断言验证关键不变量。


## 3. 二维分块同时维护分子和分母

对每个 Q tile 遍历 K/V tiles。每行维护 `m_i`、`l_i` 和未归一化输出累计 `acc_i`；最大值更新时，`l` 与 `acc` 都按同一因子缩放。最后 `O=acc/l`。下面实现不生成全局 score matrix，结果与 oracle 对齐。


In [ ]:
def tiled_attention130(q,k,v,bq=3,bk=2,causal=False):  # 定义本节可复用的核心函数。
    n,d=q.shape; out=np.empty((n,v.shape[1])); scale=math.sqrt(d)  # 计算并保存当前步骤的中间状态。
    for qs in range(0,n,bq):  # 遍历输入元素以累积或检查结果。
        qi=q[qs:qs+bq]; rows=np.arange(qs,min(qs+bq,n)); m=np.full(len(qi),-np.inf); l=np.zeros(len(qi)); acc=np.zeros((len(qi),v.shape[1]))  # 计算并保存当前步骤的中间状态。
        for ks in range(0,len(k),bk):  # 遍历输入元素以累积或检查结果。
            kj=k[ks:ks+bk]; vj=v[ks:ks+bk]; cols=np.arange(ks,min(ks+bk,len(k))); s=qi@kj.T/scale  # 计算并保存当前步骤的中间状态。
            if causal: s=np.where(cols[None,:]<=rows[:,None],s,-np.inf)  # 按当前条件选择后续控制路径。
            bm=np.max(s,axis=1); nm=np.maximum(m,bm); alpha=np.exp(m-nm); p=np.exp(s-nm[:,None])  # 计算并保存当前步骤的中间状态。
            l=l*alpha+p.sum(axis=1); acc=acc*alpha[:,None]+p@vj; m=nm  # 计算并保存当前步骤的中间状态。
        out[qs:qs+len(qi)]=acc/l[:,None]  # 计算并保存当前步骤的中间状态。
    return out  # 返回当前分支计算出的结果。
tiled130=tiled_attention130(q130,k130,v130)  # 计算并保存当前步骤的中间状态。
assert np.allclose(tiled130,attention130(q130,k130,v130),atol=1e-12)  # 用受控断言验证关键不变量。
assert tiled130.shape==(7,3)  # 用受控断言验证关键不变量。
assert np.all(np.isfinite(tiled130))  # 用受控断言验证关键不变量。


## 4. Mask 语义要进入每个 tile

causal attention 的第 `i` 行只能看 `j≤i`。不能先做局部 softmax 再把未来 token 清零，那会破坏归一化；也不能跳过包含边界的 tile。生产 kernel 会根据块位置整块跳过上三角区域，并只在对角块做元素 mask。


In [ ]:
causal_ref130=attention130(q130,k130,v130,True); causal_tile130=tiled_attention130(q130,k130,v130,2,3,True)  # 计算并保存当前步骤的中间状态。
assert np.allclose(causal_tile130,causal_ref130,atol=1e-12)  # 用受控断言验证关键不变量。
assert np.allclose(causal_ref130[0],v130[0])  # 用受控断言验证关键不变量。
changed130=v130.copy(); changed130[-1]+=1000  # 计算并保存当前步骤的中间状态。
assert np.allclose(attention130(q130,k130,changed130,True)[0],causal_ref130[0])  # 用受控断言验证关键不变量。


## 5. 节省的是中间矩阵驻留和 HBM IO

算术复杂度仍约为 `O(N²d)`，但标准训练常需保存每头 `N²` scores/probabilities；分块版本只保留 Q tile、K/V tile 和每行统计量。真实速度取决于 tile、SRAM、head dimension、序列长度、dtype 和 kernel 融合，不是所有短序列都更快。


In [ ]:
def workspace130(n,d,dv,bq,bk,bytes_=2):  # 定义本节可复用的核心函数。
    standard=n*n*bytes_*2  # 计算并保存当前步骤的中间状态。
    tiled=(bq*d+bk*d+bk*dv+bq*bk+bq*(dv+2))*bytes_  # 计算并保存当前步骤的中间状态。
    return standard,tiled  # 返回当前分支计算出的结果。
std130,tile130=workspace130(4096,128,128,128,128)  # 计算并保存当前步骤的中间状态。
assert tile130<std130  # 用受控断言验证关键不变量。
assert workspace130(8192,128,128,128,128)[0]==4*std130  # 用受控断言验证关键不变量。
assert workspace130(4096,128,128,64,64)[1]<tile130  # 用受控断言验证关键不变量。


## 6. 运行最大值是精度合同

直接 `exp(logit)` 在 FP16/FP32 都可能溢出。分块算法始终相对当前最大值指数化，并在新最大值出现时重标度旧累计量。实现还常用 FP32 累加 `m/l/acc`，即使输入是 BF16/FP16；这属于数值设计，不应为省几个寄存器随意删除。


In [ ]:
huge130=np.array([10000.,9990.,9980.]); stable130=np.exp(huge130-huge130.max()); stable130/=stable130.sum()  # 计算并保存当前步骤的中间状态。
with np.errstate(over="ignore",invalid="ignore"):  # 在受管理的上下文中执行操作。
    naive130=np.exp(huge130)/np.exp(huge130).sum()  # 计算并保存当前步骤的中间状态。
assert np.all(np.isfinite(stable130))  # 用受控断言验证关键不变量。
assert not np.all(np.isfinite(naive130))  # 用受控断言验证关键不变量。
assert math.isclose(stable130.sum(),1.0)  # 用受控断言验证关键不变量。


## 7. 重计算要求 dropout 可按坐标复现

backward 若不保存 probability，就会重算 tiles；dropout mask 必须由全局 seed、batch/head、query/key 坐标确定，而不能依赖 tile 遍历顺序。下面用哈希构造教学版 counter-based mask，证明改变分块方式不会改变同一坐标的随机决策。


In [ ]:
def keep130(seed,b,h,i,j,p=.1):  # 定义本节可复用的核心函数。
    raw=f"{seed}:{b}:{h}:{i}:{j}".encode(); u=int.from_bytes(hashlib.sha256(raw).digest()[:8],"big")/2**64  # 计算并保存当前步骤的中间状态。
    return u>=p  # 返回当前分支计算出的结果。
mask_a130=[[keep130(9,0,1,i,j) for j in range(7)] for i in range(7)]  # 计算并保存当前步骤的中间状态。
mask_b130=np.array([[keep130(9,0,1,i,j) for j in range(7)] for i in range(7)])  # 计算并保存当前步骤的中间状态。
assert np.array_equal(mask_a130,mask_b130)  # 用受控断言验证关键不变量。
assert keep130(9,0,1,2,3)==keep130(9,0,1,2,3)  # 用受控断言验证关键不变量。
assert any(not x for row in mask_a130 for x in row)  # 用受控断言验证关键不变量。


## 8. 正确性测试覆盖 tile 边界，而不只测整除形状

测试不同 Q/K block、非整除长度、causal/non-causal、大幅 logits、不同 value dimension、padding 和 dropout seed；与高精度 oracle 比较 forward/backward。性能评测同时报告 kernel time、端到端吞吐、峰值 HBM 与数值误差。


In [ ]:
variants130=[tiled_attention130(q130,k130,v130,bq,bk,True) for bq,bk in [(1,1),(2,4),(5,3),(8,8)]]  # 计算并保存当前步骤的中间状态。
assert all(np.allclose(x,causal_ref130,atol=1e-12) for x in variants130)  # 用受控断言验证关键不变量。
assert len(variants130)==4  # 用受控断言验证关键不变量。
assert max(np.max(np.abs(x-causal_ref130)) for x in variants130)<1e-10  # 用受控断言验证关键不变量。


## 面试总结

完整回答是：**标准 attention 做 oracle → 每行维护 online `m/l/acc` → Q/K/V 二维分块 → 最大值变化时重标度旧累计 → mask 在 tile 内参与归一化 → FP32 累加 → 坐标式 RNG 支持 dropout 重算 → 跨块形状验证 forward/backward → 用 HBM IO 与端到端时间评测**。FlashAttention 是 IO-aware 的精确 attention，不是把二次计算偷偷近似掉。

延伸阅读：[FlashAttention](https://arxiv.org/abs/2205.14135)、[FlashAttention-2](https://arxiv.org/abs/2307.08691)、[Online Normalizer Calculation](https://arxiv.org/abs/1805.02867)。
